<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/CNNPretrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
from google.colab import userdata,files
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

In [24]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("Kaggle API activated via Secrets!")

Kaggle API activated via Secrets!


In [26]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
140k-real-and-fake-faces.zip: Skipping, found more recently modified local copy (use --force to force download)


In [27]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

replace ./dataset_folder/real_vs_fake/real-vs-fake/test/fake/00276TOPP4.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [28]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [43]:
# train_gen = ImageDataGenerator(rescale= 1./255)
# test_gen = ImageDataGenerator(rescale= 1./255)
# validation_gen = ImageDataGenerator(rescale= 1./255)
train_gen = ImageDataGenerator(preprocessing_function=preprocess_input)
validation_gen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [44]:
train_data = train_gen.flow_from_directory(train_dir,
                                           target_size=(224,224),
                                           batch_size= 32,
                                           class_mode='binary')

test_data = train_gen.flow_from_directory(test_dir,
                                           target_size=(224,224),
                                           batch_size= 32,
                                           class_mode='binary')

validation_data = train_gen.flow_from_directory(validation_dir,
                                           target_size=(224,224),
                                           batch_size= 32,
                                           class_mode='binary')

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [34]:
# 1. Load the "Advanced" Brain (No Head, ImageNet Weights)
base_cnn = tf.keras.applications.EfficientNetV2B0(
    input_shape=(224, 224, 3),
    include_top=False, # This is the "No Head" surgery
    weights='imagenet',
    pooling='avg'      # This automatically flattens the data for us!
)

# 2. Freeze the Brain (Phase 1)
base_cnn.trainable = False

# 3. Build the Skyscraper
model = tf.keras.Sequential([
    base_cnn,
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-b0 (Functional)  │ (None, 1280)           │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,247,505 (23.83 MB)

 Trainable params: 328,193 (1.25 MB)

 Non-trainable params: 5,919,312 (22.58 MB)

In [35]:
print(base_cnn.input_shape)

(None, 224, 224, 3)


In [46]:
# 1. Use a standard Learning Rate for the new "Head"
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# 2. Launch the first 5 epochs
# I recommend setting workers=4 to speed up image loading
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=5,
    verbose=1
)

Epoch 1/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 243s 70ms/step - accuracy: 0.8091 - loss: 0.4194 - val_accuracy: 0.8529 - val_loss: 0.3360
Epoch 2/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 208s 67ms/step - accuracy: 0.8599 - loss: 0.3281 - val_accuracy: 0.8716 - val_loss: 0.2956
Epoch 3/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 200s 64ms/step - accuracy: 0.8762 - loss: 0.2942 - val_accuracy: 0.8741 - val_loss: 0.2880
Epoch 4/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 206s 66ms/step - accuracy: 0.8897 - loss: 0.2672 - val_accuracy: 0.8953 - val_loss: 0.2523
Epoch 5/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 197s 63ms/step - accuracy: 0.8973 - loss: 0.2503 - val_accuracy: 0.9048 - val_loss: 0.2302


unfreeze

In [47]:
base_cnn.trainable = True

# Let's freeze everything EXCEPT the last 20 layers
for layer in base_cnn.layers[:-20]:
    layer.trainable = False

# Now only the last 20 layers will 'learn'
optimizer = tf.keras.optimizers.Adam(learning_rate=0.00001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

In [48]:
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=5,
    verbose=1
)

Epoch 1/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 260s 74ms/step - accuracy: 0.8874 - loss: 0.2697 - val_accuracy: 0.9104 - val_loss: 0.2178
Epoch 2/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 207s 66ms/step - accuracy: 0.9127 - loss: 0.2175 - val_accuracy: 0.9224 - val_loss: 0.1872
Epoch 3/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 209s 67ms/step - accuracy: 0.9256 - loss: 0.1898 - val_accuracy: 0.9276 - val_loss: 0.1747
Epoch 4/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 205s 66ms/step - accuracy: 0.9346 - loss: 0.1676 - val_accuracy: 0.9420 - val_loss: 0.1466
Epoch 5/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 225s 72ms/step - accuracy: 0.9400 - loss: 0.1528 - val_accuracy: 0.9444 - val_loss: 0.1386
